In [ ]:
import sys
import os
import random
import numpy as np
import pandas as pd
from scipy.stats import bernoulli, norm
from sklearn import linear_model
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# ============================================================
# USER-DEFINED PROJECT PATH
# ============================================================
# IMPORTANT:
# Before running, you must define:
# base_path = "/path/to/your/CausalICM/"
#
# This folder must contain:
#   CausalICM/functions/data_simulation_multivariate.py
#   CausalICM/functions/model_training.py
#   CausalICM/functions/weightedMSE.py
#
# Example:
# base_path = "/Users/username/Research/CausalICM/"
# ============================================================

# Example (uncomment and edit):
# base_path = "/path/to/your/CausalICM/"


# Add project path to Python path
sys.path.append(base_path)

# Import internal project functions
from CausalICM.functions.data_simulation_multivariate import data_simulation4
from CausalICM.functions.model_training import ICM
from CausalICM.functions.weightedMSE import weighted_mean_squared_error


# ============================================================
# Set random seeds for reproducibility
# ============================================================
seed_value = 1234
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)


# ============================================================
# Data Simulation
# ============================================================
data_exp, data_obs = data_simulation4(
    1000, beta0=-10.0, beta1=-8.0, beta2=-8.0
)
data_full = np.r_[data_exp, data_obs]


# ============================================================
# Propensity score model for S (trial participation)
# ============================================================
logr = linear_model.LogisticRegression()
logr.fit(data_full[:, 1:4], data_full[:, 0])
prob = logr.predict_proba(data_full[:, 1:4])

prob_part = prob[:, 1]
data_full = np.c_[data_full, 1 / prob_part]   # Append weights


# ============================================================
# Hyperparameter search for rho via 5-fold CV
# ============================================================
rho_values = np.round(np.arange(0, 1.05, 0.1), 1)
num_folds = 5

kf = KFold(n_splits=num_folds, shuffle=True, random_state=seed_value)

min_avg_wmse = float("inf")
best_rho = None
wmse_per_rho = {}

for rho in rho_values:
    print(f"Rho = {rho}")
    wmse_per_fold = []

    for fold_num, (train_index, val_index) in enumerate(kf.split(data_full)):
        train_data = data_full[train_index]
        val_data = data_full[val_index]

        # Only validate on experimental samples
        val_data = val_data[val_data[:, 0] == 1, :]
        X_val = val_data[:, 1:6]
        T_val = val_data[:, 6]
        Y_val = val_data[:, 7]
        w_val = val_data[:, 8]

        # Fit ICM model
        m0, m1 = ICM(
            X_E=train_data[train_data[:, 0] == 1, 1:6],
            X_O=train_data[train_data[:, 0] == 0, 1:6],
            T_E=train_data[train_data[:, 0] == 1, 6],
            T_O=train_data[train_data[:, 0] == 0, 6],
            Y_E=train_data[train_data[:, 0] == 1, 7],
            Y_O=train_data[train_data[:, 0] == 0, 7],
            r=2,
            ID=5,
            AD=[0, 1, 2, 3, 4],
            rho=rho
        )

        # Predictions
        predY0_exp = m0.predict_noiseless(
            np.c_[X_val, np.zeros(X_val.shape[0])]
        )
        predY1_exp = m1.predict_noiseless(
            np.c_[X_val, np.zeros(X_val.shape[0])]
        )

        wmse = weighted_mean_squared_error(
            weight=w_val,
            t=T_val,
            y_true=Y_val,
            y_pred0=predY0_exp[0],
            y_pred1=predY1_exp[0]
        )

        wmse_per_fold.append(wmse)
        print(f"  Fold {fold_num + 1} WMSE: {wmse}")

    wmse_per_rho[rho] = wmse_per_fold
    avg_wmse = np.mean(wmse_per_fold)
    print(f"Average WMSE for rho = {rho}: {avg_wmse}")

    if avg_wmse < min_avg_wmse:
        min_avg_wmse = avg_wmse
        best_rho = rho


print(f"\nBest rho = {best_rho} (Min Avg WMSE = {min_avg_wmse})")


# ============================================================
# Save best rho
# ============================================================
output_path = os.path.join(base_path, "sim4_best_rho.csv")
pd.DataFrame([best_rho], columns=["best rho"]).to_csv(output_path, index=False)